# Comparación de algoritmos de aprendizaje supervisado: KNN, SVM y MLP

**Problema:** clasificación de tumores en benignos y malignos.  
**Entorno:** Google Colab · Python · scikit-learn

> Antes de ejecutar: carga el archivo del dataset en Colab y configura `TARGET_COLUMN` en la sección de configuración. Los resultados y conclusiones se generan a partir de los datos reales; no deben inventarse ni copiarse de otra ejecución.

## 1. Objetivo

Implementar y comparar K-Nearest Neighbors (KNN), Support Vector Machine (SVM) y una red neuronal artificial tipo Multi-Layer Perceptron (MLP), usando el mismo dataset, la misma partición de entrenamiento/prueba y métricas equivalentes.

## 2. Preparación del entorno

In [ ]:
!pip -q install -U scikit-learn pandas numpy matplotlib seaborn openpyxl

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.utils.multiclass import unique_labels

RANDOM_STATE = 42
TEST_SIZE = 0.20

## 3. Carga del conjunto de datos

Sube el archivo CSV o Excel proporcionado por el docente. El notebook admite `.csv`, `.xlsx` y `.xls`.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError("No se cargó ningún archivo. Vuelve a ejecutar esta celda y selecciona el dataset.")

file_name = next(iter(uploaded))
if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name)
elif file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name)
else:
    raise ValueError("Formato no compatible. Usa CSV o Excel.")

print(f"Archivo cargado: {file_name}")
display(df.head())

## 4. Exploración e identificación de variables

Revisa dimensiones, tipos de datos, valores faltantes, duplicados, columnas y clases. La variable objetivo debe ser la etiqueta que indica si el tumor es benigno o maligno.

In [ ]:
print("Dimensiones (filas, columnas):", df.shape)
print("\\nColumnas:", df.columns.tolist())
print("\\nTipos de datos:")
display(df.dtypes.to_frame("tipo de dato"))
print("\\nValores faltantes por columna:")
display(df.isna().sum().to_frame("faltantes"))
print("\\nFilas duplicadas:", df.duplicated().sum())
print("\\nResumen estadístico:")
display(df.describe(include="all").T)

### Configuración de la variable objetivo

Cambia `TARGET_COLUMN` por el nombre exacto de la columna que contiene las clases. Si el archivo incluye una columna identificadora (por ejemplo, `id`), agrégala a `DROP_COLUMNS` para que no se use como característica predictora.

In [ ]:
TARGET_COLUMN = "diagnosis"  # <-- CAMBIAR por el nombre exacto de la variable objetivo
DROP_COLUMNS = []            # Ejemplo: ["id", "Unnamed: 32"]

if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"No existe la columna objetivo '{TARGET_COLUMN}'. "
        f"Columnas disponibles: {df.columns.tolist()}. Corrige TARGET_COLUMN."
    )

data = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns]).copy()
data = data.dropna(subset=[TARGET_COLUMN]).drop_duplicates()

X = data.drop(columns=[TARGET_COLUMN]).copy()
y_raw = data[TARGET_COLUMN].copy()

# Excluir columnas identificadoras indicadas y columnas totalmente vacías.
X = X.drop(columns=[c for c in X.columns if X[c].isna().all()])

print("Cantidad de registros utilizables:", len(data))
print("Cantidad de características de entrada:", X.shape[1])
print("Variable objetivo:", TARGET_COLUMN)
print("Clases originales y frecuencias:")
display(y_raw.value_counts(dropna=False).to_frame("registros"))

## 5. Preparación y división de los datos

Se codifica la variable objetivo y se realiza una única división estratificada 80/20. Los tres modelos reciben exactamente los mismos índices de entrenamiento y prueba. La imputación, codificación de variables categóricas y estandarización se ajustan dentro de cada pipeline usando únicamente entrenamiento, evitando fuga de información.

**Importante:** en este problema interesa especialmente detectar tumores malignos. Verifica que la clase positiva identificada abajo sea realmente la clase maligna.

In [ ]:
label_encoder = LabelEncoder()
y = pd.Series(label_encoder.fit_transform(y_raw.astype(str)), index=y_raw.index, name=TARGET_COLUMN)
class_names = [str(c) for c in label_encoder.classes_]
print("Codificación de clases:", {name: int(label_encoder.transform([name])[0]) for name in class_names})

# Detección orientativa de la clase maligna para calcular precision/recall/F1 binarios.
malignant_candidates = {"m", "malignant", "maligno", "maligna", "1"}
positive_label = next(
    (int(label_encoder.transform([c])[0]) for c in class_names if c.strip().lower() in malignant_candidates),
    1 if len(class_names) == 2 else None
)
if len(class_names) != 2:
    raise ValueError("Este notebook está configurado para clasificación binaria. Se detectaron: " + str(class_names))
if positive_label is None:
    positive_label = 1

print("Clase considerada positiva para métricas binarias:", class_names[positive_label])
print("Confirma que corresponda a MALIGNO. Si no, ajusta positive_label manualmente.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Distribución de clases en entrenamiento:")
display(y_train.map(dict(enumerate(class_names))).value_counts(normalize=True).rename("proporción").to_frame())
print("Distribución de clases en prueba:")
display(y_test.map(dict(enumerate(class_names))).value_counts(normalize=True).rename("proporción").to_frame())

## 6. Preprocesamiento

Las variables numéricas se imputan con la mediana y se estandarizan. Las categóricas se imputan con la moda y se convierten mediante One-Hot Encoding. El preprocesamiento se integra en cada pipeline para evitar ajustar transformaciones con datos de prueba.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Variables numéricas:", len(numeric_features))
print("Variables categóricas:", len(categorical_features))

## 7. Entrenamiento de los tres algoritmos

Se utilizan configuraciones iniciales razonables y reproducibles. No se realiza ajuste de hiperparámetros para mantener una comparación base; si el docente solicita optimización, debe hacerse solo con validación cruzada sobre el conjunto de entrenamiento.

In [ ]:
models = {
    "KNN": KNeighborsClassifier(n_neighbors=5, weights="uniform", metric="minkowski"),
    "SVM": SVC(kernel="rbf", C=1.0, gamma="scale"),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(100,),
        activation="relu",
        solver="adam",
        max_iter=1000,
        early_stopping=True,
        random_state=RANDOM_STATE
    )
}

pipelines = {
    name: Pipeline(steps=[("preprocessing", preprocessor), ("model", model)])
    for name, model in models.items()
}

predictions = {}
for name, pipeline in pipelines.items():
    print(f"Entrenando {name}...")
    pipeline.fit(X_train, y_train)
    predictions[name] = pipeline.predict(X_test)

print("Entrenamiento y predicción completados.")

## 8. Evaluación: métricas y matrices de confusión

Se reportan Accuracy, Precision, Recall y F1-score. Para Precision, Recall y F1 se utiliza promedio binario y se considera positiva la clase maligna (revisa la configuración anterior). La matriz de confusión permite distinguir falsos negativos —tumores malignos clasificados como benignos— y falsos positivos.

In [ ]:
results = []
for name, y_pred in predictions.items():
    results.append({
        "Algoritmo": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label=positive_label, zero_division=0),
        "Recall": recall_score(y_test, y_pred, pos_label=positive_label, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, pos_label=positive_label, zero_division=0),
        "Errores": int((y_test.to_numpy() != y_pred).sum())
    })

results_df = pd.DataFrame(results).set_index("Algoritmo").sort_values("F1-score", ascending=False)
display(results_df.style.format("{:.4f}"))

for name, y_pred in predictions.items():
    print(f"\\n===== {name} =====")
    print(classification_report(
        y_test, y_pred, labels=[0, 1], target_names=class_names, zero_division=0
    ))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Matriz de confusión — {name}")
    plt.tight_layout()
    plt.show()

## 9. Gráfica comparativa de rendimiento

In [ ]:
ax = results_df[["Accuracy", "Precision", "Recall", "F1-score"]].plot(
    kind="bar", figsize=(11, 6), ylim=(0, 1), rot=0
)
ax.set_title("Comparación de métricas por algoritmo")
ax.set_ylabel("Puntuación")
ax.set_xlabel("Algoritmo")
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

ax = results_df["Errores"].plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_title("Cantidad de errores en el conjunto de prueba")
ax.set_ylabel("Predicciones incorrectas")
ax.set_xlabel("Algoritmo")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Interpretación automática de resultados

La siguiente celda resume qué modelo lidera cada criterio. Usa este resultado como apoyo y redacta la interpretación final considerando las matrices de confusión y el costo de los errores.

In [ ]:
best_accuracy = results_df["Accuracy"].idxmax()
best_f1 = results_df["F1-score"].idxmax()
best_recall = results_df["Recall"].idxmax()
most_errors = results_df["Errores"].idxmax()
least_errors = results_df["Errores"].idxmin()

print(f"Mayor Accuracy: {best_accuracy} ({results_df.loc[best_accuracy, 'Accuracy']:.4f})")
print(f"Mayor F1-score: {best_f1} ({results_df.loc[best_f1, 'F1-score']:.4f})")
print(f"Mayor Recall para la clase positiva: {best_recall} ({results_df.loc[best_recall, 'Recall']:.4f})")
print(f"Más errores: {most_errors} ({results_df.loc[most_errors, 'Errores']})")
print(f"Menos errores: {least_errors} ({results_df.loc[least_errors, 'Errores']})")

for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    print(f"{name}: TN={tn}, FP={fp}, FN={fn}, TP={tp} | Falsos negativos malignos={fn}")

## 11. Conclusiones y preguntas de la actividad

Completa la interpretación después de revisar las métricas y matrices. No afirmes que un modelo es mejor solo por Accuracy.

1. **¿Cuál obtuvo el mejor rendimiento?** Compara Accuracy, F1-score y, especialmente, Recall de la clase maligna. Indica el modelo y sus valores.
2. **¿Cuál presentó más errores?** Señala el algoritmo con mayor valor en `Errores` y compara sus matrices de confusión.
3. **¿Qué diferencias observaste?** Describe cómo varían los falsos positivos, falsos negativos y las métricas entre KNN, SVM y MLP.
4. **¿Elegirías el algoritmo con mayor exactitud?** Justifica. En una clasificación relacionada con tumores, un falso negativo (tumor maligno clasificado como benigno) puede ser especialmente grave. Por ello, el Recall de la clase maligna y el equilibrio reflejado por F1-score pueden ser más relevantes que Accuracy por sí sola. La decisión real requiere validación clínica y no sustituye el diagnóstico médico.

### Plantilla para redactar la conclusión final

> En este experimento, el algoritmo ______ obtuvo el mayor Accuracy de ______, mientras que ______ alcanzó el mayor Recall para la clase maligna (______). El modelo ______ presentó más errores, con ______ clasificaciones incorrectas. Las diferencias observadas muestran que ______. Para este problema no elegiríamos automáticamente el modelo con mayor exactitud: priorizaríamos ______ porque ______. También consideraríamos los falsos negativos, ya que ______. Estos resultados corresponden a esta partición de prueba y pueden variar con otra muestra o configuración.

## 12. Reproducibilidad y entrega

- Ejecutar todas las celdas en orden en Google Colab.
- Confirmar que no haya errores y que las métricas/gráficas se hayan generado.
- Guardar el notebook con las salidas: **Archivo → Guardar una copia en GitHub** o descargar `.ipynb` y subirlo al repositorio.
- Incluir el dataset solo si la licencia y las instrucciones del curso lo permiten. Si no, documentar su fuente y cómo cargarlo.
- Revisar que el repositorio sea público o que el docente tenga acceso.